# W01 Environment Check

Verifies the Week 1 research toolchain: Python 3.11, CUDA-enabled PyTorch, HuggingFace `transformers`/`datasets`/Hub, PEFT, LangChain, Weights & Biases, NumPy, pandas, and SciPy. The checks avoid external credentials and model downloads so the notebook can run on a fresh clone with the local environment installed.

In [1]:
import sys
import platform

print(sys.version)
print(platform.platform())
assert sys.version_info[:2] == (3, 11), f"Expected Python 3.11, got {sys.version_info[:3]}"

3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
Windows-10-10.0.26200-SP0


In [2]:
import torch

print("torch", torch.__version__)
print("torch_cuda", torch.version.cuda)
print("cuda_available", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA is required for this research environment"

device = torch.device("cuda")
print("device_count", torch.cuda.device_count())
print("device_name", torch.cuda.get_device_name(0))

x = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device=device)
y = x @ x
torch.cuda.synchronize()
print(y.cpu())
assert y.is_cuda
assert y.shape == (2, 2)

torch 2.11.0+cu128
torch_cuda 12.8
cuda_available True
device_count 1
device_name NVIDIA GeForce RTX 3080 Ti Laptop GPU


tensor([[ 7., 10.],
        [15., 22.]])


In [3]:
import numpy as np
import pandas as pd
import scipy
from scipy import stats

values = np.array([1.0, 2.0, 3.0])
frame = pd.DataFrame({"value": values})
t_stat, p_value = stats.ttest_1samp(values, popmean=2.0)

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scipy", scipy.__version__)
print(frame.describe())
print("ttest", t_stat, p_value)
assert frame["value"].mean() == 2.0
assert np.isfinite(t_stat) or np.isnan(t_stat)

numpy 2.4.6
pandas 3.0.3
scipy 1.17.1
       value
count    3.0
mean     2.0
std      1.0
min      1.0
25%      1.5
50%      2.0
75%      2.5
max      3.0
ttest 0.0 1.0


In [4]:
import transformers
from datasets import Dataset
from huggingface_hub import HfApi

ds = Dataset.from_dict({"text": ["physical ai", "pic 2.0"], "label": [1, 1]})
api = HfApi()

print("transformers", transformers.__version__)
print("dataset rows", len(ds))
print("hub api", type(api).__name__)
assert len(ds) == 2

C:\Users\IHAVE\Python Projects\InGen\.conda-w01\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers 5.12.0
dataset rows 2
hub api HfApi


In [5]:
import peft
from peft import LoraConfig, TaskType

cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=4, lora_alpha=8, lora_dropout=0.0)
print("peft", peft.__version__)
print(cfg)
assert cfg.r == 4

peft 0.19.1
LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=4, target_modules=None, exclude_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [6]:
import langchain
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

doc = Document(page_content="PIC 2.0 maps physical AI model classes to open literature.", metadata={"week": 1})
prompt = ChatPromptTemplate.from_messages([("system", "You are a research assistant."), ("human", "Summarize: {text}")])
messages = prompt.format_messages(text=doc.page_content)

print("langchain", langchain.__version__)
print(messages[-1].content)
assert "PIC 2.0" in messages[-1].content

langchain 1.3.9
Summarize: PIC 2.0 maps physical AI model classes to open literature.


In [7]:
import os
import wandb

os.environ["WANDB_MODE"] = "disabled"
run = wandb.init(project="ingen-w01-env-check", mode="disabled")
wandb.log({"env_check": 1})
wandb.finish()

print("wandb", wandb.__version__)
assert run is not None

wandb 0.27.2


## Result

If every cell above completes, the Week 1 research environment is ready for CUDA-backed PyTorch checks, lightweight literature/RAG work, fine-tuning configuration, statistics/evaluation work, and experiment-tracking smoke tests.